In [ ]:
# adapt -> we need to give some input and ask AI to learn from those training examples and then predict the blank in test data accordingly.

import google.generativeai as genai
from google.colab import userdata
import os
import time  # ADDED

# API KEY
# we are using LLM api key here. loaded inside the var Api_KEY
API_KEY = userdata.get('API_KEY')
genai.configure(api_key=API_KEY)

Xtrain = []
trainData = [
    {"ipsentence":"Let's schedule a team <MASK> to discuss the project.", "maskedword":"meeting"},
    {"ipsentence":"The little black dress is <MASK>", "maskedword":"expensive"},
    {"ipsentence":"I will <MASK> my parents tomorrow.", "maskedword": "visit"}
]

testData = [
    {"ipsentence":"He is not feeling <MASK>"},
    {"ipsentence":"They were <MASK> badminton yesterday."},
    {"ipsentence":"I will be <MASK> to Europe."}
]

# Load the model here usingb
model = genai.GenerativeModel("gemini-2.5-pro")  # gemini-2.5-pro

def testDataPrediction(testsentence, inputPrompt):
    inputPrompt += f"Now predict the masked word for = "
    inputPrompt += f"{testsentence}"
    inputPrompt += f"<MASK>:"
    return inputPrompt

# init inp prompt empty
inputPrompt = ""

# input prompt involving prompt that allows AI to adapt to the train data tell it everything specifically add
# inputPrompt += f"Some examples of input sentences are provided for you below."
inputPrompt += f"Consider the training data examples and missing word <MASK>"


inputPrompt += f"Training Sentences and MASK word: "
# for each sentence present in training data
for data in trainData:
    # for training data we need inp sentence and the MASKED word.
    # then the model will learn and adapt
    inputPrompt += f"Input Sentence = {data['ipsentence']}"
    inputPrompt += f"Maskedword: {data['maskedword']} "

# try to adapt and test for test data
inputPrompt += f"After learning from input now predict the mask word for test sentences"
# inputPrompt += f"and predict the most appropriate word that fits the <MASK> for every sentence in the test data."


# test data prompt
for i, testsentence in enumerate(testData):
    # current_prompt = ""
    print(f"Sent test {i+1}: {testsentence}")

    # freshly generated promp every test case
    current_prompt = inputPrompt + f" Now predict the masked word for: {testsentence['ipsentence']} <MASK>:"

    # get response
    response = model.generate_content(current_prompt)
    # get only text bec it can be in other format jsoni think
    print(f"AIResp: {response.text}\n")

    # was getting too many requests api issue
    time.sleep(2)

print("*AI Resp Generation done*")

Sent test 1: {'ipsentence': 'He is not feeling <MASK>'}
AIResp: well

Sent test 2: {'ipsentence': 'They were <MASK> badminton yesterday.'}
AIResp: playing

Sent test 3: {'ipsentence': 'I will be <MASK> to Europe.'}
AIResp: traveling

*AI Resp Generation done*


In [23]:
# Question-1 - Build and Pretrain neural network models using Pytorch


import torch
import random

####################################### Part-1 PREPROCESSING #######################################
# Data pre-processing: You have to preprocess the data (both training and testing) and
# create a vocabulary of unique words, # transform all words into lower-case. # The input word embedding can be simply one-hot encoding. N × V .
# For the [MASK] token, the corresponding embedding is a vector of length V of all zeros

# Part 1 -> reading and cleaning the data
numWords = {}
def read_data(file_name):
    file = open(file_name, "r")
    content = file.read()
    file.close()
    return content

def is_alphabet(ch):
    if ch >= 'a' and ch <= 'z':
        return True
    return ch >= 'A' and ch <= 'Z'

def clean_content(content):
    # Convert to string to handle [MASK] replacement properly
    if isinstance(content, list):
      content_str = ''.join(content)
    else:
      content_str = content

    # Replace [MASK] with MASKEDWORD. if we dont replace we were getting issues with [ ad ]
    content_str = content_str.replace("[MASK]", "MASKEDWORD")

    # Convert back to list for character processing
    content_list = list(content_str)

    for idx in range(len(content_list)):
        if not is_alphabet(content_list[idx]):    # replace spaces, commas etc whatever is not an alphabet with an _ in content
            content_list[idx] = '_'               # return content. content is a list this has a list of characters not special symbols.
    return content_list

# Part 2 -> Construct the vocabulary
def get_words(content):
  # print()
  words = []
  i = 0
  while i < len(content):					# run loop through every character in content
    if content[i] == '_':         # if _ is encountered in content then ignore it and increment i.
      i += 1
      continue
    word = ''
    j = i
    while j < len(content):				# This loop is to capture the word between two underscore characters
      if content[j] != '_':				# if content[j] is not _
        word += content[j]				# save it as word. So we read all the characters till we encounter a _ and save the whole thing as one word.
        j += 1										# this makes sense because every word is preceded and ended with a space (could have comma or something)
      else:
        break
    # print(word)
    i += len(word)					   			 # i will now point to the start of next word.
    words.append(word.lower())		   # collect all the words. CONVERtING EVERYTHING TO LOWER. HEnce not Romeo
  # print("\nWords = "+str(words))
  return words

# Part-3 wordtoIndex
def formVocab(words):
  vocab = set(words)
  print(vocab)
  # vocab.discard("maskedword")      # if maskedword is present remove else keep quiet dont error
  return vocab

def wordtoindex(trainingwords):
  # form
  totalwords = list(formVocab(trainingwords))
  totalWordSize = len(totalwords)
  wordToIndexDict = {}

  for index, word in enumerate(totalwords):
    wordToIndexDict[word] = index
  return totalwords, totalWordSize, wordToIndexDict

# obtain sentences
def obtainIndividualSentences(data):
  sentences = []
  for sent in data.splitlines():
    sentence = sent.strip()
    sentences.append(sentence)
  # print("\nsentences = "+str(sentences))
  return sentences

# we ar trying to represnt words as vectors through One hot encoding
def oneHotEncoding(words, wordtoIndex, vocabSize):
  # wordtoIndex = converts words to index. Assigns index to each word.
  oneHotEncoding = []
  for word in words:
    # since initial is all zeros <MASK> will have all zeros
    vocabVect = torch.zeros(vocabSize)    # initial vocabvect is of all zeros
    if word in wordtoIndex:
      if word != 'maskedword':               # if u encounter MASK continue
        # wordtoIndex[word] gives you index of a specific word.
        # in vocabVect that specific index of word will get value 1 remaining 0
        vocabVect[wordtoIndex[word]] = 1
    oneHotEncoding.append(vocabVect)
  if oneHotEncoding:                     # Check if list is not empty
    ohe = torch.stack(oneHotEncoding)
  else:
    ohe = torch.tensor([])

  print(f"Matrix shape: {ohe.shape}")
  print(f"Number of 1's: {torch.sum(ohe == 1).item()}")
  print(f"Number of 0's: {torch.sum(ohe == 0).item()}")
  return ohe

# MLM MASKED SENTENCE GENERATON
def MLMmask():
  # storing masked Sentences for training. will have sentences with mask at various pos and the corresponding words
  with open("training_data.txt", "r") as trainFile:
      with open("maskedSentences.txt", "w") as maskFile:
          # trainingData = read_data("training_data.txt")
          # testData = read_data("testing_data.txt")

          # read the data from train  save in trainingData
          trainingData = trainFile.readlines()
          # testData = testFile.readlines()
          # totalData = trainingData + testData
          totalData = trainingData
          print("\nTraining Data = "+str(totalData))

          data1 = []
          maskedPositions = []

          for sentnce in totalData:
              # we have to generate 15 exmples from the above one single sentence.
              # So for each sentence we will have 15 * 8 examples total = 120
              print("\nSentnce = "+str(sentnce))

              for count in range(15):
                  # we will get one by one sentences from totalData
                  sentence = list(sentnce)
                  maskedWords = []
                  # perform cleaning and everything of the data like converting all into lower and replacing extra characters by underscore.
                  # sentences are cleaned and we will get words from the sentences.
                  cleaned_content = clean_content(sentence)
                  wordsOfSentence = get_words(cleaned_content)
                  wordsOfSentencewithMask = wordsOfSentence.copy()   # extract word from current sentence

                  # pick one randomint # this is mainly to kno how many words to be masked
                  if len(wordsOfSentence) > 0:
                      # how many masks do we want to add is this maskCount
                      maskCount = random.randint(1, min(4, len(wordsOfSentence)))
                      maskPositions = random.sample(range(len(wordsOfSentence)), maskCount)  # getlist of masked positions for a specific sentence
                      print("mask positions = "+str(maskPositions))

                  # we assign mask to random positions in the sentence and this is done continuously on one sentence.
                  # keep adding at diff positions save those in the file.

                  for position in maskPositions:
                      if position < len(wordsOfSentence):
                          maskedWords.append(wordsOfSentence[position])
                          wordsOfSentencewithMask[position] = "maskedword"

                  sentenceWithMask = " ".join(wordsOfSentencewithMask)

                  # saving in masked sentences file
                  # we are saving maskedSentence, then maskedWords and their positions also bcz we need it for training na
                  maskFile.write(sentenceWithMask + " -> " +str(maskedWords)+ " -> " +str(maskPositions)+ "\n")



print("**Data pre-processing and Masked Sentences Obtained**")

# 1 Training the data diff steps perform like cleaning data, getting words
trainingData = read_data("training_data.txt")
# print("trainingData = "+str(trainingData))
trainingcontent = list(read_data("training_data.txt"))    # read text
trainingcontent = clean_content(trainingcontent)
trainingwords = get_words(trainingcontent)
print("\nTraining words = "+str(trainingwords))

# Test
testingData = read_data("testing_data.txt")
testingcontent = list(read_data("testing_data.txt"))
# testingcontent = ["i","love","programming","[MASK]","python"]
testingcontent = clean_content(testingcontent)
testingwords = get_words(testingcontent)
print("\nTesting words = "+str(testingwords))

# we need One Hot Encoding matrIx for trainingwords and testing words
# 1. For training  # 1. we need wordtoIndex
totalVocab, vocabSize, wordToIndexDict = wordtoindex(trainingwords)

# 2. now we need to do one hot for train and fone hot for test
# 2.a Train data one hot enc
trainSentences_EncodingMatricesList = []
testSentences_EncodingMatricesList = []
# first we need to get sentences individually inorder to form N * V matrix
individualSentencesTrain = obtainIndividualSentences(trainingData)
individualSentencesTest = obtainIndividualSentences(testingData)

# 3. getting our one hot vectorS for training and testing
# 3. a Train data
print("\n** Train **")
for index, sent in enumerate(individualSentencesTrain):
    cleanSentence = clean_content(list(sent))
    sentWords = get_words(cleanSentence)
    print("\nSentence  = "+str(sent))
    print("Sentence after cleaning = "+str(cleanSentence))
    sentMatrix = oneHotEncoding(sentWords, wordToIndexDict, vocabSize)
    trainSentences_EncodingMatricesList.append(sentMatrix)

# 3.b Test data
print("\n** Test **")
for index, testsent in enumerate(individualSentencesTest):
    cleanSentenceTest = clean_content(list(testsent))
    testsentWords = get_words(cleanSentenceTest)
    print("\nSentence  = "+str(testsent))
    print("Sentence after cleaning = "+str(cleanSentenceTest))
    testsentMatrix = oneHotEncoding(testsentWords, wordToIndexDict, vocabSize)
    testSentences_EncodingMatricesList.append(testsentMatrix)


# Code Part 2: MLM Masked Sentence Generation
print("MLM Mask Generation")
MLMmask()

**Data pre-processing and Masked Sentences Obtained**

Training words = ['i', 'teach', 'computer', 'science', 'in', 'the', 'department', 'of', 'computer', 'science', 'at', 'university', 'of', 'alabama', 'at', 'birmingham', 'i', 'am', 'a', 'professor', 'teaching', 'artificial', 'intelligence', 'in', 'the', 'department', 'of', 'computer', 'science', 'at', 'university', 'of', 'alabama', 'at', 'birmingham', 'i', 'am', 'a', 'student', 'at', 'university', 'of', 'alabama', 'at', 'birmingham', 'i', 'study', 'computer', 'science', 'as', 'a', 'student', 'at', 'university', 'of', 'alabama', 'at', 'birmingham', 'a', 'professor', 'and', 'a', 'student', 'are', 'at', 'university', 'of', 'alabama', 'at', 'birmingham', 'she', 'is', 'a', 'nurse', 'working', 'in', 'the', 'school', 'of', 'medicine', 'he', 'is', 'a', 'doctor', 'working', 'in', 'the', 'school', 'of', 'medicine', 'a', 'doctor', 'and', 'a', 'nurse', 'work', 'in', 'the', 'school', 'of', 'medicine']

Testing words = ['in', 'the', 'department', 

In [22]:
######## LSTM model###############

# efine LSTM neural network architecture here. this is the main model class
# checking fwd ani backwrd and then we are predicting the next word by considering both the contexts.
# we are placing masks at random places so both contexts should be nexessary and important for predition.

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim

class LSTMModelClass(nn.Module):
  # here initialising the LSTM mdoel
  # embeddingSize = size of wordvectors, hidden_size = hiddensize(LSTM), dropout = prevents regularization
  def __init__(self, vocabSiz, inputSize, hiddenSize, num_layers = 2, dropout=0.3):

    super(LSTMModelClass, self).__init__()
    # change/map frm word to vector of fixed size.  # similar words = similar embedding vectors and they are clse to each other when plotted in the map
    # inputSize = size(inp) we get this one from the embedding layer

    # 1. Word to Vec kela
    self.embedding = nn.Embedding(vocabSiz, inputSize)

    # hiddenSize = hidden unit in LSTM cels # num_layers = 2 LSTM layers. #  layer1= basix pattern , 2ndlayer = highlevel patern
    # dropout -> for regularization prevent overfitting     # bidirectional -> both way LSTm, fwd , bckwd
    # fwd = The mat on , backwrd -> on mat The     # (sizeofbatch, sequenceLenth, hiddenSize*2)
    self.lstm = nn.LSTM(inputSize, hiddenSize, num_layers, batch_first=True, dropout=dropout, bidirectional=True)

    # apply activation and get output = LSTm op. Then applying dropout means some of the outputs make it zero
    self.dropout = nn.Dropout(dropout)

    # Since LSTM is bidirectional, hidden size is doubled for the fully connected layer
    self.fc = nn.Linear(hiddenSize * 2, vocabSiz)

  def forward(self, x):
    # x = word rep as tensor
    embedding = self.embedding(x)
    # LSTM layer processing
    lstm_out, hidden = self.lstm(embedding)
    # apply dropout to LSTM output
    lstm_out = self.dropout(lstm_out)
    # fully connected layer to get final output
    output = self.fc(lstm_out)
    return output


def prepTrainData(wordToIndexDict):
    with open("maskedSentences.txt", "r") as maskedFile:
        maxsentLen = 50
        sentencesTensor = []
        maskWordIndices = []
        # maskedWordsFinal = []
        # maskedPositionsFinal = []
        # outDataFinal = []
        # postoActualWord = []
        # sentIndexWithPadding = []

        for line in maskedFile:
            # split line into diff parts
            sections = line.strip().split(" -> ")

            words = sections[0].split()
            maskedWords = eval(sections[1])        # ['in','of']
            maskedPositions = eval(sections[2])    #  [4,7]

            indexIP = []
            indexOP = []
            # word to index mapping
            for ind, word in enumerate(words):
                # check if word is masked word
                if word != "maskedword":
                    indexIP.append(wordToIndexDict.get(word, 0))
                    indexOP.append(-10000)

                else:
                    # get index of mask token and append to input index
                    indexIP.append(0)  # mask token
                    pos = maskedPositions.index(ind)
                    indexOP.append(wordToIndexDict[maskedWords[pos]])

            zeroExtLen = maxsentLen - len(indexIP)
            indexIP.extend([0] * zeroExtLen)
            indexOP.extend([-100] * zeroExtLen)
            # sentencesInp will ahve
            sentencesTensor.append(indexIP)
            maskWordIndices.append(indexOP)

    # ele = list of integers, int = index(word) from wordToIndexDict
    sentencesTensor = torch.tensor(sentencesTensor, dtype=torch.long) # sentence represented by numbers. MASK = 0
    maskWordIndices = torch.tensor(maskWordIndices, dtype=torch.long)

    return sentencesTensor, maskWordIndices


def LSTM_onTrainData():
  learningRate = 0.001   # default lr for Adam optimizer
  epochs = 30

  sentencesFinal, maskedWordsIndices  = prepTrainData(wordToIndexDict)
  # we want pyt tensor not py list of sentences
  sentencesFinal = torch.tensor(sentencesFinal, dtype= torch.long)
  # we want datasettt insuch a way that the lstm has the data gets the data easy
  datasetLSTM = TensorDataset(sentencesFinal)
  # the batchprocessor processes the data batch by batch
  batchProcessor = DataLoader(datasetLSTM, batch_size = 20, shuffle = True)
  vocabWordSize = len(wordToIndexDict)

  # create model
  lstm = LSTMModelClass(vocabWordSize, inputSize = 64, hiddenSize = 128)

  # for adam we know we need less tuning and then it is faster thanother optim. converges fast
  # optimizer always updates weight so odel learnsand gives less loss. model = lstm
  optimizerRes = optim.Adam(lstm.parameters(), lr = learningRate)
  # crossentropy weneed for classification or predicting the next word. similar words together checks prob of each words before finalising
  lossObt = nn.CrossEntropyLoss()
  lossFinal = 0
  accurateTotal = 0
  predTotal = 0

  for epoch in range(epochs):

      lossFinal = 0
      accurateTotal = 0
      predTotal = 0
      # get each batch ikde each oer iteration
      for index, indBatch in enumerate(batchProcessor):
          # get the tesor
          batchData = indBatch[0]

          # Reset gradients here otherise the gradients that were calcualted will get accumulated
          optimizerRes.zero_grad()

          # Forward passperform lstm
          lstmPred = lstm(batchData)  # Shape: (20, 12, vocabWordSize)

          # obtaining the loss after lstm execution here loss cal ho jave
          lossObtained = lossObt(lstmPred.view(-1, vocabWordSize), batchData.view(-1))
          # calc gradietns and update the weight
          lossObtained.backward()
          optimizerRes.step()

          # Calculate accuracy
          with torch.no_grad():

              predIndexs = torch.argmax(lstmPred, dim=2)
              accuratePrediction = (predIndexs == batchData)

              # Count correct predictions
              accurateTotal += accuratePrediction.float().sum().item()
              predTotal += batchData.numel()

          lossFinal += lossObtained.item()

      # gte average loss and find accuraccy
      Lossavg = lossFinal / len(batchProcessor)
      accuracyObtained = accurateTotal / predTotal

      print(f"Epoch {epoch+1}/{epochs}, Loss: {Lossavg:.4f}, Accuracy: {accuracyObtained:.4f}")

  torch.save(lstm.state_dict(), "trainedmodellstm.pth")
  print("*******Training completed********")
  print("\n")

################# TEST DATA PART ##################
# preparation of test data the cleaning and everything is performed in preprocessing code aboce so
# our testdata is already having sentences with [MASK] token.





print("***** LSTM Prediction on Training Data *****")
LSTM_onTrainData()
# LSTM_onTestData()

***** LSTM Prediction on Training Data *****


/tmp/ipython-input-489797884.py:101: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sentencesFinal = torch.tensor(sentencesFinal, dtype= torch.long)


Epoch 1/30, Loss: 2.7588, Accuracy: 0.6717
Epoch 2/30, Loss: 1.0096, Accuracy: 0.8093
Epoch 3/30, Loss: 0.9662, Accuracy: 0.8093
Epoch 4/30, Loss: 0.7776, Accuracy: 0.8132
Epoch 5/30, Loss: 0.7317, Accuracy: 0.8228
Epoch 6/30, Loss: 0.6992, Accuracy: 0.8280
Epoch 7/30, Loss: 0.6614, Accuracy: 0.8297
Epoch 8/30, Loss: 0.6331, Accuracy: 0.8343
Epoch 9/30, Loss: 0.6016, Accuracy: 0.8428
Epoch 10/30, Loss: 0.5678, Accuracy: 0.8545
Epoch 11/30, Loss: 0.5323, Accuracy: 0.8605
Epoch 12/30, Loss: 0.4971, Accuracy: 0.8637
Epoch 13/30, Loss: 0.4597, Accuracy: 0.8740
Epoch 14/30, Loss: 0.4186, Accuracy: 0.8907
Epoch 15/30, Loss: 0.3756, Accuracy: 0.9098
Epoch 16/30, Loss: 0.3296, Accuracy: 0.9243
Epoch 17/30, Loss: 0.2844, Accuracy: 0.9385
Epoch 18/30, Loss: 0.2403, Accuracy: 0.9477
Epoch 19/30, Loss: 0.1986, Accuracy: 0.9578
Epoch 20/30, Loss: 0.1618, Accuracy: 0.9705
Epoch 21/30, Loss: 0.1324, Accuracy: 0.9762
Epoch 22/30, Loss: 0.1060, Accuracy: 0.9825
Epoch 23/30, Loss: 0.0861, Accuracy: 0.98